# 🍎 Ejercicios: Apple
# App Store - 1.200 apps (precio, rating, descargas)

⚠️ Datos SIMULADOS / ilustrativos
(generados con pandas + numpy).
No son cifras oficiales del App Store.

### 💼 PREGUNTA ENTREVISTA:
### ¿Las apps gratis se
### descargan mucho más
### que las de pago?

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/app_store_apps.csv'
)

r = (
  df
  .groupby('es_gratis')
  .agg(
    descargas_med=('descargas',
                   'median'),
    rating_med=('rating', 'median'),
    apps=('app', 'count'),
  )
  .round(1)
)

ratio = (
  r.loc[True, 'descargas_med']
  / r.loc[False, 'descargas_med']
)

print(r)
print(f'\nLas gratis x{ratio:.1f} descargas')

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué categorías mueven
### más descargas y cuáles
### valoran mejor?

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/app_store_apps.csv'
)

r = (
  df
  .groupby('categoria')
  .agg(
    descargas_M=('descargas', 'sum'),
    rating=('rating', 'mean'),
    apps=('app', 'count'),
  )
)

r['descargas_M'] = (
  r['descargas_M'] / 1e6
).round(1)
r['rating'] = r['rating'].round(2)

print('Por categoría:')
print(
  r.sort_values('descargas_M',
    ascending=False)
)

### 💼 PREGUNTA ENTREVISTA:
### ¿Cuánto factura cada
### app de pago?  (precio
### x descargas, top 10)

Apple se queda su comisión de todo
lo que facturan las apps de pago.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/app_store_apps.csv'
)

pago = df[~df['es_gratis']].copy()

pago['ingreso_bruto'] = (
  pago['precio_usd']
  * pago['descargas']
)

# Comisión Apple 30%
pago['corte_apple'] = (
  pago['ingreso_bruto'] * 0.30
)

top = (
  pago
  .nlargest(10, 'ingreso_bruto')
  [['app', 'categoria',
    'precio_usd', 'corte_apple']]
)

top['corte_apple'] = (
  (top['corte_apple'] / 1e6).round(2)
)

print('Top 10 (corte Apple en M USD):')
print(top.to_string(index=False))

### 💼 PREGUNTA ENTREVISTA:
### ¿Un buen rating depende
### del precio de la app?

¿La gente perdona más a las apps
caras o las castiga más duro?

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/app_store_apps.csv'
)

df['franja'] = pd.cut(
  df['precio_usd'],
  bins=[-0.1, 0, 2.99, 9.99, 100],
  labels=['Gratis', 'Barata',
          'Media', 'Cara'],
)

r = (
  df
  .groupby('franja', observed=True)
  .agg(
    rating=('rating', 'mean'),
    apps=('app', 'count'),
  )
  .round(2)
)

print('Rating por franja de precio:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿Las apps más pesadas
### se valoran peor?

Ocupan espacio en el iPhone: ¿lo
nota el usuario en su valoración?

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/app_store_apps.csv'
)

df['peso'] = pd.qcut(
  df['tamano_mb'],
  q=4,
  labels=['Ligera', 'Normal',
          'Pesada', 'Enorme'],
)

r = (
  df
  .groupby('peso', observed=True)
  .agg(
    rating=('rating', 'mean'),
    mb=('tamano_mb', 'mean'),
  )
  .round(2)
)

corr = (
  df['tamano_mb'].corr(df['rating'])
)

print(r)
print(f'\nCorrelación peso-rating: {corr:.3f}')